# 02 — Inference (Kaggle, self-contained)

Run this **second** (after `01_train_lora.ipynb`), or run it alone with `HF_ADAPTER_REPO = None` for base-model predictions.

Clones the repo from GitHub, loads Qwen2.5-VL-7B (4-bit Unsloth) + optional LoRA adapter from HuggingFace Hub, runs sync detect → transcribe on the competition test set, writes `submission.csv`.

**Setup:**
- Right sidebar → _+ Add Data_ → attach `handwritten-to-data` (the competition).
- Internet: On.
- If your adapter repo is private: Kaggle → Add-ons → Secrets → label `HF_TOKEN` → **toggle Attached on for this notebook** → restart kernel.

Nothing to edit unless you want a different config or to skip the adapter.

In [ ]:
# 1. Install GPU stack.
%pip -q install --upgrade unsloth unsloth_zoo
%pip -q install --no-deps trl peft accelerate bitsandbytes
%pip -q install pyyaml opencv-python-headless

In [ ]:
GITHUB_REPO     = "https://github.com/dhodyrev/handwritten-to-data.git"
HF_ADAPTER_REPO = "dkhodyriev/htr-qwen25vl-transcribe-lora"  # set to None for base-model inference
CONFIG          = "configs/pipeline.yaml"                     # or configs/pipeline_p1.yaml

In [ ]:
# 3. Clone repo into /kaggle/working/repo and install htr.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)
print("cwd:", os.getcwd())

In [ ]:
# 4. HuggingFace login — only needed if the adapter repo is private.
if HF_ADAPTER_REPO:
    from huggingface_hub import login
    try:
        from kaggle_secrets import UserSecretsClient
        login(token=UserSecretsClient().get_secret("HF_TOKEN"))
        print("hf login OK")
    except Exception as e:
        print(f"hf login skipped ({e}); proceeding anonymously (only works if adapter is public)")

In [ ]:
# 5. Build a test manifest from sample_submission.csv. Source is unknown for
#    held-out pages, so the pipeline falls back to the generic block-then-line
#    path (no math short-circuit).
import csv, json, pathlib
TEST_DIR = "/kaggle/input/handwritten-to-data/test"
SAMPLE   = "/kaggle/input/handwritten-to-data/sample_submission.csv"
manifest = pathlib.Path("data/cv/test.jsonl")
manifest.parent.mkdir(parents=True, exist_ok=True)
with open(SAMPLE) as f, open(manifest, "w") as out:
    n = 0
    for row in csv.DictReader(f):
        out.write(json.dumps({"file_name": row["image"], "source": "unknown"}) + "\n")
        n += 1
print(f"wrote {n} test items to {manifest}")

In [ ]:
# 6. Run inference. backend.load_model passes the adapter string through to
#    model.load_adapter() which resolves HF Hub IDs the same as local paths.
adapter_arg = f"--adapter {HF_ADAPTER_REPO}" if HF_ADAPTER_REPO else ""
!python scripts/run_inference.py \
    --manifest data/cv/test.jsonl \
    --image-root {TEST_DIR} \
    --config {CONFIG} \
    {adapter_arg} \
    --out /kaggle/working/submission.csv

In [ ]:
# 7. Spot-check the submission.
import csv
with open("/kaggle/working/submission.csv") as f:
    rows = list(csv.DictReader(f))
print(f"rows: {len(rows)}")
print("first image:", rows[0]["image"])
print("regions head:", rows[0]["regions"][:300], "...")